**Instalacion de librerias y Chrome Driver**

In [ ]:
import requests, subprocess, zipfile, io, os

# Instalar Chrome directamente desde Google
subprocess.run(["wget", "-q", "https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "./google-chrome-stable_current_amd64.deb"], check=True)
subprocess.run(["pip", "install", "-q", "selenium"], check=True)

# Descarga de ChromeDriver para la versión de Chrome instalada
chrome_version = subprocess.check_output(
    ["google-chrome", "--version"]
).decode().split()[2]

url = f"https://storage.googleapis.com/chrome-for-testing-public/{chrome_version}/linux64/chromedriver-linux64.zip"
zip_file = zipfile.ZipFile(io.BytesIO(requests.get(url).content))
zip_file.extractall()
os.system("mv chromedriver-linux64/chromedriver /usr/local/bin/ && chmod +x /usr/local/bin/chromedriver")
print(f"ChromeDriver {chrome_version} instalado")

ChromeDriver 148.0.7778.178 instalado


**Parseo del HTML (primeros pasos)**

In [ ]:


apartamentos = []

#1. TÍTULO
titulo = soup.find("h1", attrs={"tabindex": "-1"})
titulo_text = titulo.text.strip() if titulo else "N/A"

#2. PRECIO TOTAL
precio = soup.find("span", attrs={"style":"--pricing-guest-primary-line-unit-price-text-decoration: none;"})
precio_noche = float(precio.get_text().replace("\xa0", "").replace("€", ""))

#3. VIAJEROS

wait = WebDriverWait(driver, 10)

elementos = wait.until(
    EC.presence_of_all_elements_located((By.XPATH, "//li[contains(text(), 'viajero')]"))
)

num_viajeros = int(re.search(r'\d+', elementos[0].text).group())


#4. DORMITORIOS

elementos = wait.until(
    EC.presence_of_all_elements_located((By.XPATH, "//li[contains(text(), 'dormitorio')]"))
)

num_dormitorios = int(re.search(r'\d+', elementos[0].text).group())

#5. CAMAS

elementos = wait.until(
    EC.presence_of_all_elements_located((By.XPATH, "//li[contains(text(), 'cama')]"))
)

num_camas = int(re.search(r'\d+', elementos[0].text).group())

#6. BAÑOS

elementos = wait.until(
    EC.presence_of_all_elements_located((By.XPATH, "//li[contains(text(), 'baño')]"))
)

num_baños = int(re.search(r'\d+', elementos[0].text).group())

#7. NOTA MEDIA Y Nº DE RESEÑAS

elemento = wait.until(
    EC.presence_of_element_located((By.XPATH, "//span[contains(text(), 'valoración')]"))
)

texto = elemento.get_attribute("innerHTML")

nota = float(re.search(r'(\d+[,\.]\d+)', texto).group().replace(',', '.'))
num_reseñas = int(re.search(r'(\d+)\s*(?:&nbsp;)?evaluaciones', texto).group(1))

#9. APPEND AL DICCIONARIO

apartamentos.append({
    "titulo":       titulo_text,
    "precio_noche": precio_noche,
    "viajeros": num_viajeros,
    "dormitorios": num_dormitorios,
    "camas": num_camas,
    "baños": num_baños,
    "nota": nota,
    "nº reseñas": num_reseñas,
})

apartamentos

[{'titulo': 'Apto. céntrico y silencioso',
  'precio_noche': 633.0,
  'viajeros': 4,
  'dormitorios': 2,
  'camas': 2,
  'baños': 1,
  'nota': 5.0,
  'nº reseñas': 5}]

**Script definitivo (nivel distrito Madrid)**

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
import re
import time
import csv


CHECKIN = "2026-06-01"
CHECKOUT = "2026-06-02"

CIUDAD = "Madrid"

DISTRITOS = [
    "Tetuán",
    "Carabanchel",
    "Usera",
    "Valdebebas",
    "Puente-de-Vallecas",
    "Chamberí"
]


def construir_url(distrito):

    distrito_url = distrito.replace(" ", "-")

    return (
        f"https://www.airbnb.es/s/"
        f"{distrito_url}--Madrid/homes"
        f"?refinement_paths%5B%5D=%2Fhomes"
        f"&date_picker_type=calendar"
        f"&checkin={CHECKIN}"
        f"&checkout={CHECKOUT}"
        f"&search_type=autocomplete_click"
    )


def scroll_completo(driver):

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight)"
    )

    time.sleep(2)

    driver.execute_script(
        "window.scrollTo(0, 0)"
    )

    time.sleep(1)

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight)"
    )

    time.sleep(2)


def extraer_n(driver, xpath):

    try:

        elementos = driver.find_elements(By.XPATH, xpath)

        if elementos:

            return int(
                re.search(r'\d+', elementos[0].text).group()
            )

    except:
        pass

    return None


def extraer_detalle_apartamento(driver, url):

    driver.get(
        url + f"?checkin={CHECKIN}&checkout={CHECKOUT}"
    )

    try:

        wait.until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    "//div[@data-section-id='OVERVIEW_DEFAULT_V2']"
                )
            )
        )

    except:
        pass

    time.sleep(2)

    viajeros = extraer_n(
        driver,
        "//li[contains(text(), 'viajero')]"
    )

    dormitorios = extraer_n(
        driver,
        "//li[contains(text(), 'dormitorio')]"
    )

    camas = extraer_n(
        driver,
        "//li[contains(text(), 'cama')]"
    )

    baños = extraer_n(
        driver,
        "//li[contains(text(), 'baño')]"
    )

    return viajeros, dormitorios, camas, baños


def extraer_cards(soup, datos_cards, distrito):

    cards = soup.find_all(
        attrs={"data-testid": "card-container"}
    )

    urls_existentes = {
        d["url"] for d in datos_cards
    }

    for card in cards:

        a = card.find(
            "a",
            href=lambda h: h and h.startswith("/rooms/")
        )

        if not a:
            continue

        url = (
            "https://www.airbnb.es"
            + a["href"].split("?")[0]
        )

        if url in urls_existentes:
            continue

        room_id = None

        for tag in [card] + card.find_all(True):

            lid = (
                tag.get("data-listing-id")
                or tag.get("data-id")
                or tag.get("data-room-id")
            )

            if lid and re.fullmatch(r'\d+', str(lid).strip()):

                room_id = str(lid).strip()
                break

        if not room_id:

            m_id = re.search(
                r'/rooms/(?:[^/\?]+/)?(\d{6,})',
                a["href"]
            )

            if m_id:
                room_id = m_id.group(1)

        titulo = None

        titulo_tag = card.find(
            "div",
            attrs={"data-testid": "listing-card-title"}
        )

        if titulo_tag:
            titulo = titulo_tag.get_text(strip=True)

        precio = None

        btn = card.find(
            "button",
            attrs={"aria-expanded": "false"}
        )

        if btn:

            for span in btn.find_all("span"):

                texto = (
                    span.get_text(strip=True)
                    .replace("\xa0", " ")
                )

                m = re.search(r'^(\d+)\s*€', texto)

                if m:
                    precio = float(m.group(1))
                    break

        nota = None
        reseñas = None

        for span in card.find_all(
            "span",
            {"aria-hidden": "true"}
        ):

            m = re.match(
                r'([\d,\.]+)\s*\((\d+)\)',
                span.get_text().strip()
            )

            if m:

                nota = float(
                    m.group(1).replace(",", ".")
                )

                reseñas = int(m.group(2))

                break

        datos_cards.append({

            "ciudad": CIUDAD,
            "distrito": distrito,
            "room_id": room_id,
            "fecha_entrada": CHECKIN,
            "url": url,
            "titulo": titulo,
            "precio": precio,

            "viajeros": None,
            "dormitorios": None,
            "camas": None,
            "baños": None,

            "nota": nota,
            "reseñas": reseñas,
        })

        urls_existentes.add(url)

    return len(cards)


opts = Options()

opts.add_argument("--headless")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=opts)

wait = WebDriverWait(driver, 15)


campos = [
    "ciudad",
    "distrito",
    "room_id",
    "fecha_entrada",
    "titulo",
    "precio",
    "viajeros",
    "dormitorios",
    "camas",
    "baños",
    "nota",
    "reseñas",
    "url"
]


todos_los_datos = []


for DISTRITO in DISTRITOS:

    print(f"\nSCRAPEANDO: {DISTRITO}")

    URL_BUSQUEDA = construir_url(DISTRITO)

    datos_cards = []

    driver.get(URL_BUSQUEDA)

    wait.until(
        EC.presence_of_all_elements_located(
            (
                By.XPATH,
                "//div[@data-testid='card-container']"
            )
        )
    )

    time.sleep(3)

    scroll_completo(driver)

    for pagina in range(1, 16):

        soup = BeautifulSoup(
            driver.page_source,
            "html.parser"
        )

        extraer_cards(
            soup,
            datos_cards,
            DISTRITO
        )

        print(
            f"{DISTRITO} | "
            f"Pagina {pagina} | "
            f"{len(datos_cards)} apartamentos"
        )

        if pagina == 15:
            break

        try:

            siguiente = wait.until(
                EC.presence_of_element_located(
                    (
                        By.XPATH,
                        "//a[@aria-label='Siguiente']"
                    )
                )
            )

            siguiente_url_raw = (
                siguiente.get_attribute("href")
            )

            parsed = urlparse(siguiente_url_raw)

            params = parse_qs(parsed.query)

            cursor = params.get("cursor", [None])[0]

            if cursor:

                siguiente_url = (
                    URL_BUSQUEDA
                    + f"&cursor={cursor}"
                )

            else:

                siguiente_url = siguiente_url_raw

            driver.get(siguiente_url)

            time.sleep(5)

            wait.until(
                EC.presence_of_all_elements_located(
                    (
                        By.XPATH,
                        "//div[@data-testid='card-container']"
                    )
                )
            )

            scroll_completo(driver)

        except:
            break

    print("\nExtrayendo detalles...\n")

    for i, d in enumerate(datos_cards):

        print(f"[{i+1}/{len(datos_cards)}] {d['titulo']}")

        viajeros, dormitorios, camas, baños = (
            extraer_detalle_apartamento(
                driver,
                d["url"]
            )
        )

        d["viajeros"] = viajeros
        d["dormitorios"] = dormitorios
        d["camas"] = camas
        d["baños"] = baños

        time.sleep(1)

    nombre_csv = (
        "apartamentos_"
        + DISTRITO.lower().replace(" ", "_")
        + ".csv"
    )

    with open(
        nombre_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=campos,
            quoting=csv.QUOTE_NONNUMERIC
        )

        writer.writeheader()
        writer.writerows(datos_cards)

    print(f"\nCSV guardado: {nombre_csv}")

    todos_los_datos.extend(datos_cards)


nombre_csv_final = "apartamentos_madrid_total.csv"

with open(
    nombre_csv_final,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=campos,
        quoting=csv.QUOTE_NONNUMERIC
    )

    writer.writeheader()
    writer.writerows(todos_los_datos)

print(f"\nCSV FINAL guardado: {nombre_csv_final}")

driver.quit()

print("\nSCRAPING FINALIZADO")


SCRAPEANDO: Tetuán
Tetuán | Pagina 1 | 24 apartamentos
Tetuán | Pagina 2 | 52 apartamentos
Tetuán | Pagina 3 | 70 apartamentos
Tetuán | Pagina 4 | 88 apartamentos
Tetuán | Pagina 5 | 105 apartamentos
Tetuán | Pagina 6 | 123 apartamentos
Tetuán | Pagina 7 | 141 apartamentos
Tetuán | Pagina 8 | 159 apartamentos
Tetuán | Pagina 9 | 177 apartamentos
Tetuán | Pagina 10 | 195 apartamentos
Tetuán | Pagina 11 | 213 apartamentos
Tetuán | Pagina 12 | 231 apartamentos
Tetuán | Pagina 13 | 249 apartamentos
Tetuán | Pagina 14 | 267 apartamentos
Tetuán | Pagina 15 | 285 apartamentos

Extrayendo detalles...

[1/285] Apartamento en Madrid
[2/285] Alojamiento en Tetuán
[3/285] Alojamiento en Tetuán
[4/285] Apartamento en Tetuán
[5/285] Apartamento en Cuatro Caminos
[6/285] Apartamento en Tetuán
[7/285] Habitación en Tetuán
[8/285] Habitación en Tetuán
[9/285] Apartamento en Tetuán
[10/285] Habitación en Tetuán
[11/285] Habitación en Tetuán
[12/285] Habitación en Chamberí
[13/285] Alojamiento en Madrid

Script definitivo (nivel distrito Barcelona)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
import re
import time
import csv


CHECKIN = "2026-06-01"
CHECKOUT = "2026-06-02"

CIUDAD = "Barcelona"

DISTRITOS = [
    "Les-Corts",
    "Poblenou",
    "Sants-montjuic",
    "Gracia",
    "Eixample",
    "Horta~Guinardó"
]


def construir_url(distrito):

    distrito_url = distrito.replace(" ", "-")

    return (
        f"https://www.airbnb.es/s/"
        f"{distrito_url}--{CIUDAD}/homes"
        f"?refinement_paths%5B%5D=%2Fhomes"
        f"&date_picker_type=calendar"
        f"&checkin={CHECKIN}"
        f"&checkout={CHECKOUT}"
        f"&search_type=autocomplete_click"
    )


def scroll_completo(driver):

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight)"
    )

    time.sleep(2)

    driver.execute_script(
        "window.scrollTo(0, 0)"
    )

    time.sleep(1)

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight)"
    )

    time.sleep(2)


def extraer_n(driver, xpath):

    try:

        elementos = driver.find_elements(By.XPATH, xpath)

        if elementos:

            return int(
                re.search(r'\d+', elementos[0].text).group()
            )

    except:
        pass

    return None


def extraer_detalle_apartamento(driver, url):

    driver.get(
        url + f"?checkin={CHECKIN}&checkout={CHECKOUT}"
    )

    try:

        wait.until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    "//div[@data-section-id='OVERVIEW_DEFAULT_V2']"
                )
            )
        )

    except:
        pass

    time.sleep(2)

    viajeros = extraer_n(
        driver,
        "//li[contains(text(), 'viajero')]"
    )

    dormitorios = extraer_n(
        driver,
        "//li[contains(text(), 'dormitorio')]"
    )

    camas = extraer_n(
        driver,
        "//li[contains(text(), 'cama')]"
    )

    baños = extraer_n(
        driver,
        "//li[contains(text(), 'baño')]"
    )

    return viajeros, dormitorios, camas, baños


def extraer_cards(soup, datos_cards, distrito):

    cards = soup.find_all(
        attrs={"data-testid": "card-container"}
    )

    urls_existentes = {
        d["url"] for d in datos_cards
    }

    for card in cards:

        a = card.find(
            "a",
            href=lambda h: h and h.startswith("/rooms/")
        )

        if not a:
            continue

        url = (
            "https://www.airbnb.es"
            + a["href"].split("?")[0]
        )

        if url in urls_existentes:
            continue

        room_id = None

        for tag in [card] + card.find_all(True):

            lid = (
                tag.get("data-listing-id")
                or tag.get("data-id")
                or tag.get("data-room-id")
            )

            if lid and re.fullmatch(r'\d+', str(lid).strip()):

                room_id = str(lid).strip()
                break

        if not room_id:

            m_id = re.search(
                r'/rooms/(?:[^/\?]+/)?(\d{6,})',
                a["href"]
            )

            if m_id:
                room_id = m_id.group(1)

        titulo = None

        titulo_tag = card.find(
            "div",
            attrs={"data-testid": "listing-card-title"}
        )

        if titulo_tag:
            titulo = titulo_tag.get_text(strip=True)

        precio = None

        btn = card.find(
            "button",
            attrs={"aria-expanded": "false"}
        )

        if btn:

            for span in btn.find_all("span"):

                texto = (
                    span.get_text(strip=True)
                    .replace("\xa0", " ")
                )

                m = re.search(r'^(\d+)\s*€', texto)

                if m:
                    precio = float(m.group(1))
                    break

        nota = None
        reseñas = None

        for span in card.find_all(
            "span",
            {"aria-hidden": "true"}
        ):

            m = re.match(
                r'([\d,\.]+)\s*\((\d+)\)',
                span.get_text().strip()
            )

            if m:

                nota = float(
                    m.group(1).replace(",", ".")
                )

                reseñas = int(m.group(2))

                break

        datos_cards.append({

            "ciudad": CIUDAD,
            "distrito": distrito,
            "room_id": room_id,
            "fecha_entrada": CHECKIN,
            "url": url,
            "titulo": titulo,
            "precio": precio,

            "viajeros": None,
            "dormitorios": None,
            "camas": None,
            "baños": None,

            "nota": nota,
            "reseñas": reseñas,
        })

        urls_existentes.add(url)

    return len(cards)


opts = Options()

opts.add_argument("--headless")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=opts)

wait = WebDriverWait(driver, 15)


campos = [
    "ciudad",
    "distrito",
    "room_id",
    "fecha_entrada",
    "titulo",
    "precio",
    "viajeros",
    "dormitorios",
    "camas",
    "baños",
    "nota",
    "reseñas",
    "url"
]


todos_los_datos = []


for DISTRITO in DISTRITOS:

    print(f"\nSCRAPEANDO: {DISTRITO}")

    URL_BUSQUEDA = construir_url(DISTRITO)

    datos_cards = []

    driver.get(URL_BUSQUEDA)

    wait.until(
        EC.presence_of_all_elements_located(
            (
                By.XPATH,
                "//div[@data-testid='card-container']"
            )
        )
    )

    time.sleep(3)

    scroll_completo(driver)

    for pagina in range(1, 16):

        soup = BeautifulSoup(
            driver.page_source,
            "html.parser"
        )

        extraer_cards(
            soup,
            datos_cards,
            DISTRITO
        )

        print(
            f"{DISTRITO} | "
            f"Pagina {pagina} | "
            f"{len(datos_cards)} apartamentos"
        )

        if pagina == 15:
            break

        try:

            siguiente = wait.until(
                EC.presence_of_element_located(
                    (
                        By.XPATH,
                        "//a[@aria-label='Siguiente']"
                    )
                )
            )

            siguiente_url_raw = (
                siguiente.get_attribute("href")
            )

            parsed = urlparse(siguiente_url_raw)

            params = parse_qs(parsed.query)

            cursor = params.get("cursor", [None])[0]

            if cursor:

                siguiente_url = (
                    URL_BUSQUEDA
                    + f"&cursor={cursor}"
                )

            else:

                siguiente_url = siguiente_url_raw

            driver.get(siguiente_url)

            time.sleep(5)

            wait.until(
                EC.presence_of_all_elements_located(
                    (
                        By.XPATH,
                        "//div[@data-testid='card-container']"
                    )
                )
            )

            scroll_completo(driver)

        except:
            break

    print("\nExtrayendo detalles...\n")

    for i, d in enumerate(datos_cards):

        print(f"[{i+1}/{len(datos_cards)}] {d['titulo']}")

        viajeros, dormitorios, camas, baños = (
            extraer_detalle_apartamento(
                driver,
                d["url"]
            )
        )

        d["viajeros"] = viajeros
        d["dormitorios"] = dormitorios
        d["camas"] = camas
        d["baños"] = baños

        time.sleep(1)

    nombre_csv = (
        "apartamentos_"
        + DISTRITO.lower().replace(" ", "_")
        + ".csv"
    )

    with open(
        nombre_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=campos,
            quoting=csv.QUOTE_NONNUMERIC
        )

        writer.writeheader()
        writer.writerows(datos_cards)

    print(f"\nCSV guardado: {nombre_csv}")

    todos_los_datos.extend(datos_cards)


nombre_csv_final = "apartamentos_barcelona_total.csv"

with open(
    nombre_csv_final,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=campos,
        quoting=csv.QUOTE_NONNUMERIC
    )

    writer.writeheader()
    writer.writerows(todos_los_datos)

print(f"\nCSV FINAL guardado: {nombre_csv_final}")

driver.quit()

print("\nSCRAPING FINALIZADO")


SCRAPEANDO: Les-Corts
Les-Corts | Pagina 1 | 28 apartamentos
Les-Corts | Pagina 2 | 51 apartamentos
Les-Corts | Pagina 3 | 68 apartamentos
Les-Corts | Pagina 4 | 84 apartamentos
Les-Corts | Pagina 5 | 101 apartamentos
Les-Corts | Pagina 6 | 118 apartamentos
Les-Corts | Pagina 7 | 136 apartamentos
Les-Corts | Pagina 8 | 153 apartamentos
Les-Corts | Pagina 9 | 171 apartamentos
Les-Corts | Pagina 10 | 189 apartamentos
Les-Corts | Pagina 11 | 207 apartamentos
Les-Corts | Pagina 12 | 225 apartamentos
Les-Corts | Pagina 13 | 243 apartamentos
Les-Corts | Pagina 14 | 261 apartamentos
Les-Corts | Pagina 15 | 278 apartamentos

Extrayendo detalles...

[1/278] Apartamento en Barcelona
[2/278] Habitación de hotel compartida en Barcelona
[3/278] Apartamento en Barcelona
[4/278] Apartamento en Barcelona
[5/278] Alojamiento en Barcelona
[6/278] Habitación en L'Hospitalet de Llobregat
[7/278] Tres Torres Atiram Hotels
[8/278] Hotel Ronda Lesseps
[9/278] Sercotel Porta Barcelona
[10/278] chic&basic Vel

Script definitivo (nivel distrito Valencia)

In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from urllib.parse import urlparse, parse_qs
import re
import time
import csv


CHECKIN = "2026-06-01"
CHECKOUT = "2026-06-02"

CIUDAD = "Valencia"

DISTRITOS = [
    "Rascanya",
    "Campanar",
    "L'Olivereta",
    "Patraix",
    "Benicalap",
    "Cuatre-Carreres"
]


def construir_url(distrito):

    distrito_url = distrito.replace(" ", "-")

    return (
        f"https://www.airbnb.es/s/"
        f"{distrito_url}--{CIUDAD}/homes"
        f"?refinement_paths%5B%5D=%2Fhomes"
        f"&date_picker_type=calendar"
        f"&checkin={CHECKIN}"
        f"&checkout={CHECKOUT}"
        f"&search_type=autocomplete_click"
    )


def scroll_completo(driver):

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight)"
    )

    time.sleep(2)

    driver.execute_script(
        "window.scrollTo(0, 0)"
    )

    time.sleep(1)

    driver.execute_script(
        "window.scrollTo(0, document.body.scrollHeight)"
    )

    time.sleep(2)


def extraer_n(driver, xpath):

    try:

        elementos = driver.find_elements(By.XPATH, xpath)

        if elementos:

            return int(
                re.search(r'\d+', elementos[0].text).group()
            )

    except:
        pass

    return None


def extraer_detalle_apartamento(driver, url):

    driver.get(
        url + f"?checkin={CHECKIN}&checkout={CHECKOUT}"
    )

    try:

        wait.until(
            EC.presence_of_element_located(
                (
                    By.XPATH,
                    "//div[@data-section-id='OVERVIEW_DEFAULT_V2']"
                )
            )
        )

    except:
        pass

    time.sleep(2)

    viajeros = extraer_n(
        driver,
        "//li[contains(text(), 'viajero')]"
    )

    dormitorios = extraer_n(
        driver,
        "//li[contains(text(), 'dormitorio')]"
    )

    camas = extraer_n(
        driver,
        "//li[contains(text(), 'cama')]"
    )

    baños = extraer_n(
        driver,
        "//li[contains(text(), 'baño')]"
    )

    return viajeros, dormitorios, camas, baños


def extraer_cards(soup, datos_cards, distrito):

    cards = soup.find_all(
        attrs={"data-testid": "card-container"}
    )

    urls_existentes = {
        d["url"] for d in datos_cards
    }

    for card in cards:

        a = card.find(
            "a",
            href=lambda h: h and h.startswith("/rooms/")
        )

        if not a:
            continue

        url = (
            "https://www.airbnb.es"
            + a["href"].split("?")[0]
        )

        if url in urls_existentes:
            continue

        room_id = None

        for tag in [card] + card.find_all(True):

            lid = (
                tag.get("data-listing-id")
                or tag.get("data-id")
                or tag.get("data-room-id")
            )

            if lid and re.fullmatch(r'\d+', str(lid).strip()):

                room_id = str(lid).strip()
                break

        if not room_id:

            m_id = re.search(
                r'/rooms/(?:[^/\?]+/)?(\d{6,})',
                a["href"]
            )

            if m_id:
                room_id = m_id.group(1)

        titulo = None

        titulo_tag = card.find(
            "div",
            attrs={"data-testid": "listing-card-title"}
        )

        if titulo_tag:
            titulo = titulo_tag.get_text(strip=True)

        precio = None

        btn = card.find(
            "button",
            attrs={"aria-expanded": "false"}
        )

        if btn:

            for span in btn.find_all("span"):

                texto = (
                    span.get_text(strip=True)
                    .replace("\xa0", " ")
                )

                m = re.search(r'^(\d+)\s*€', texto)

                if m:
                    precio = float(m.group(1))
                    break

        nota = None
        reseñas = None

        for span in card.find_all(
            "span",
            {"aria-hidden": "true"}
        ):

            m = re.match(
                r'([\d,\.]+)\s*\((\d+)\)',
                span.get_text().strip()
            )

            if m:

                nota = float(
                    m.group(1).replace(",", ".")
                )

                reseñas = int(m.group(2))

                break

        datos_cards.append({

            "ciudad": CIUDAD,
            "distrito": distrito,
            "room_id": room_id,
            "fecha_entrada": CHECKIN,
            "url": url,
            "titulo": titulo,
            "precio": precio,

            "viajeros": None,
            "dormitorios": None,
            "camas": None,
            "baños": None,

            "nota": nota,
            "reseñas": reseñas,
        })

        urls_existentes.add(url)

    return len(cards)


opts = Options()

opts.add_argument("--headless")
opts.add_argument("--no-sandbox")
opts.add_argument("--disable-dev-shm-usage")
opts.add_argument("--window-size=1920,1080")

driver = webdriver.Chrome(options=opts)

wait = WebDriverWait(driver, 15)


campos = [
    "ciudad",
    "distrito",
    "room_id",
    "fecha_entrada",
    "titulo",
    "precio",
    "viajeros",
    "dormitorios",
    "camas",
    "baños",
    "nota",
    "reseñas",
    "url"
]


todos_los_datos = []


for DISTRITO in DISTRITOS:

    print(f"\nSCRAPEANDO: {DISTRITO}")

    URL_BUSQUEDA = construir_url(DISTRITO)

    datos_cards = []

    driver.get(URL_BUSQUEDA)

    wait.until(
        EC.presence_of_all_elements_located(
            (
                By.XPATH,
                "//div[@data-testid='card-container']"
            )
        )
    )

    time.sleep(3)

    scroll_completo(driver)

    for pagina in range(1, 16):

        soup = BeautifulSoup(
            driver.page_source,
            "html.parser"
        )

        extraer_cards(
            soup,
            datos_cards,
            DISTRITO
        )

        print(
            f"{DISTRITO} | "
            f"Pagina {pagina} | "
            f"{len(datos_cards)} apartamentos"
        )

        if pagina == 15:
            break

        try:

            siguiente = wait.until(
                EC.presence_of_element_located(
                    (
                        By.XPATH,
                        "//a[@aria-label='Siguiente']"
                    )
                )
            )

            siguiente_url_raw = (
                siguiente.get_attribute("href")
            )

            parsed = urlparse(siguiente_url_raw)

            params = parse_qs(parsed.query)

            cursor = params.get("cursor", [None])[0]

            if cursor:

                siguiente_url = (
                    URL_BUSQUEDA
                    + f"&cursor={cursor}"
                )

            else:

                siguiente_url = siguiente_url_raw

            driver.get(siguiente_url)

            time.sleep(5)

            wait.until(
                EC.presence_of_all_elements_located(
                    (
                        By.XPATH,
                        "//div[@data-testid='card-container']"
                    )
                )
            )

            scroll_completo(driver)

        except:
            break

    print("\nExtrayendo detalles...\n")

    for i, d in enumerate(datos_cards):

        print(f"[{i+1}/{len(datos_cards)}] {d['titulo']}")

        viajeros, dormitorios, camas, baños = (
            extraer_detalle_apartamento(
                driver,
                d["url"]
            )
        )

        d["viajeros"] = viajeros
        d["dormitorios"] = dormitorios
        d["camas"] = camas
        d["baños"] = baños

        time.sleep(1)

    nombre_csv = (
        "apartamentos_"
        + DISTRITO.lower().replace(" ", "_")
        + ".csv"
    )

    with open(
        nombre_csv,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.DictWriter(
            f,
            fieldnames=campos,
            quoting=csv.QUOTE_NONNUMERIC
        )

        writer.writeheader()
        writer.writerows(datos_cards)

    print(f"\nCSV guardado: {nombre_csv}")

    todos_los_datos.extend(datos_cards)


nombre_csv_final = "apartamentos_valencia_total.csv"

with open(
    nombre_csv_final,
    "w",
    newline="",
    encoding="utf-8"
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=campos,
        quoting=csv.QUOTE_NONNUMERIC
    )

    writer.writeheader()
    writer.writerows(todos_los_datos)

print(f"\nCSV FINAL guardado: {nombre_csv_final}")

driver.quit()

print("\nSCRAPING FINALIZADO")


SCRAPEANDO: Rascanya
Rascanya | Pagina 1 | 24 apartamentos
Rascanya | Pagina 2 | 42 apartamentos
Rascanya | Pagina 3 | 60 apartamentos
Rascanya | Pagina 4 | 78 apartamentos
Rascanya | Pagina 5 | 96 apartamentos
Rascanya | Pagina 6 | 114 apartamentos
Rascanya | Pagina 7 | 132 apartamentos
Rascanya | Pagina 8 | 150 apartamentos
Rascanya | Pagina 9 | 168 apartamentos
Rascanya | Pagina 10 | 186 apartamentos
Rascanya | Pagina 11 | 204 apartamentos
Rascanya | Pagina 12 | 222 apartamentos
Rascanya | Pagina 13 | 240 apartamentos
Rascanya | Pagina 14 | 258 apartamentos
Rascanya | Pagina 15 | 276 apartamentos

Extrayendo detalles...

[1/276] Loft en Valencia
[2/276] Habitación en Extramurs
[3/276] Apartamento en Valencia
[4/276] Alojamiento en Benicalap
[5/276] Apartamento en Patraix
[6/276] Alojamiento en Extramurs
[7/276] Alojamiento en L'Eixample
[8/276] Apartamento en Valencia
[9/276] Apartamento en Valencia
[10/276] Loft en Valencia
[11/276] Loft en Ciutat Vella
[12/276] Apartamento en Ciu